# Running Veyrat's ORIGAMI

Runs Veyrat's `subdivide`-branch ORIGAMI C code from Python on a small test simulation, then reads the crossing-number (CN) tags it writes.

In [1]:
import subprocess
import os 
import numpy as np

Folder holding the compiled ORIGAMI executables (`origamidiv`, `origamicomb`, `origamihalo`, …)

In [2]:
BIN = "../origami_veyrat/code"

## Helpers

- **`run_args(prog, *args)`** runs one ORIGAMI program with the given command-line arguments (`origamidiv`, `origamicomb`, or just a parameter file for the others) and prints its output. `OMP_NUM_THREADS` sets how many cores OpenMP uses (really only matters if `nsplit` > 1).

- **`read_tags(path)`** reads `tag.dat`: an int32 particle count, then one byte per particle (0 void, 1 wall, 2 filament, 3 halo).

In [3]:
def run_args(prog, *args, threads=8):
    env = {**os.environ, "OMP_NUM_THREADS": str(threads)}
    run = subprocess.run([f"{BIN}/{prog}", *map(str, args)], env=env,
                       capture_output=True, text=True)
    print(run.stdout)
    if run.stderr:
        print(run.stderr)
    if run.returncode != 0:     # 1 = ORIGAMI error, negative = killed by a signal
        raise RuntimeError(f"{prog} failed (exit code {run.returncode})")

def read_tags(path):
    with open(path, "rb") as f:
        n = np.fromfile(f, np.int32, 1)[0]
        return np.fromfile(f, np.uint8, n)


## Make a test particle file

ORIGAMI needs particles that **started on a regular grid** and then moved. Where neighbouring particles pass or "cross" each other, ORIGAMI counts a crossing.

- **`make_test_walls`** Particles in 45 ≤ x ≤ 55 oscillate about x = 50 on a spring, so they cross only along x. Every particle in that slab should get CN = 1, and everything else CN = 0. With N = 64 that's 7 of 64 x-slices, so the expected CN = 1 fraction is 7/64 ≈ 0.109.
- Particle order must be `index = ix + iy·N + iz·N²` — ORIGAMI infers each particle's starting grid point from its position in the list. `order="F"` does this.

In [4]:
def make_test_walls(path, N=64, L=100.0, seed=42):
    rng = np.random.default_rng(seed)
    dx = L/N #spacing between particles 
    q1 = np.arange(N) * dx 
    
    #make N^3 cube of particles and store their x,y,z in qx, qy, qz
    qx, qy, qz = np.meshgrid(q1, q1, q1, indexing='ij') 


    #flattened starting grids, fortran order used b/c of how origami reads grid: x + yN + zN^2
    x0 = qx.ravel(order="F") 
    y0 = qy.ravel(order="F") 
    z0 = qz.ravel(order="F")

    #attract particles in 45<x<55 to x=50  with spring-like force 
    window = (qx >= 45) & (qx <= 55) 
    
    #initalizing lists to evolve 
    x = qx.copy() 
    vx = np.zeros_like(qx)

    #feel free to adjust 
    dt = 0.01
    omega = 1
    nsteps = 1000
    
    for n in range(nsteps):
        ax = -omega**2 * (x - 50) #spring force with equil. at x=50 
        ax[~window] = 0 #0 accel outside desired window 
        vx += ax*dt
        x += vx*dt 

    x = x % L #wrap x into the box 
    x1 = x.ravel(order="F") #flatten final positions (no evolution in y,z no need to flatten)

    with open(path, "wb") as f:
        np.array([N**3], dtype=np.int64).tofile(f)
        x1.astype(np.float32).tofile(f)
        y0.astype(np.float32).tofile(f)
        z0.astype(np.float32).tofile(f)
    
make_test_walls("../running_new_ORIGAMI/1CN_n64_div.pos")

## Run `origamidiv` + `origamicomb`

Computes each particle's crossing number one sub-cube at a time, then stitches the sub-cubes into a single `outdir + taglabel + "tag.dat"` (same format `origamihalo` reads). Settings come from `parameters_example.txt`; `NDIV` here is the number of divisions per side (`NDIV`³ sub-cubes).

- **1) `origamidiv params NDIV divid`** tags sub-cube `divid` plus a buffer of `nbuf` grid cells on each side, and saves only the sub-cube as `<label>_tag_<divid>.dat`.
- **2) `origamicomb label outdir np1d NDIV`** reads those `<label>_tag_<divid>.dat` files directly and reorders the sub-cubes back into ID order. Requires `np1d` divisible by `NDIV`³.

In [6]:
PARAMS = "parameters_example.txt"
OUT, LABEL, NP1D, NDIV = "../runs/test/", "veyrat_test1_n64", 64, 2

# 1) one origamidiv run per sub-cube
# Remember that div requires the PARAMS file, 
# this file should point to the particle position file created above 

for divid in range(NDIV**3):
    run_args("origamidiv", PARAMS, NDIV, divid)

# 2) stitch the sub-cubes together w/ comb 
run_args("origamicomb", LABEL, OUT, NP1D, NDIV)


Reading parameter file parameters_example.txt
262144 particles
Calculating ORIGAMI morphology.
1
64
nhalo=0,0,0,0,0
Writing morphology file to ../runs/test/veyrat_test1_n64_tag_0.dat.

Reading parameter file parameters_example.txt
262144 particles
Calculating ORIGAMI morphology.
1
64
nhalo=0,0,0,0,0
Writing morphology file to ../runs/test/veyrat_test1_n64_tag_1.dat.

Reading parameter file parameters_example.txt
262144 particles
Calculating ORIGAMI morphology.
1
64
nhalo=0,0,0,0,0
Writing morphology file to ../runs/test/veyrat_test1_n64_tag_2.dat.

Reading parameter file parameters_example.txt
262144 particles
Calculating ORIGAMI morphology.
1
64
nhalo=0,0,0,0,0
Writing morphology file to ../runs/test/veyrat_test1_n64_tag_3.dat.

Reading parameter file parameters_example.txt
262144 particles
Calculating ORIGAMI morphology.
1
64
nhalo=0,0,0,0,0
Writing morphology file to ../runs/test/veyrat_test1_n64_tag_4.dat.

Reading parameter file parameters_example.txt
262144 particles
Calculating 

## Read the results

`np.bincount` counts particles with tag 0, 1, 2, 3; dividing by the total gives the **fraction in each CN class** `[void, wall, filament, halo]`.

In [7]:
tag = read_tags(f"{OUT}{LABEL}tag.dat")
print(np.bincount(tag, minlength=4) / tag.size)

[0.890625 0.109375 0.       0.      ]
